# InterviewIQ — Final AI Model Training

**Architecture:** Separate Question/Answer representations → Early Fusion & Late Fusion → MLP Neural Classifier → Weak / Average / Strong.

This notebook evaluates four lightweight text representation approaches on the supplied validated dataset.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'InterviewIQ_Validated_Master_Dataset.csv'
df = pd.read_csv(DATA)
df.head()

,Dataset_ID,Batch,Role,Question,Answer,Validator_1,Validator_2,Validator_3,Final_Label,Validation_Agreement
0,IQ-0001,1,Software Developer,"For a Software Developer interview, how would ...",I would explain object-oriented programming cl...,Average,Strong,Average,Average,Majority
1,IQ-0002,1,Software Developer,"For a Software Developer interview, how would ...",I know APIs is important in this role. It help...,Strong,Average,Average,Average,Majority
2,IQ-0003,1,Software Developer,"For a Software Developer interview, how would ...",I have heard of authentication and authorizati...,Strong,Average,Strong,Strong,Majority
3,IQ-0004,1,Software Developer,"For a Software Developer interview, how would ...","I would explain Git version control clearly, c...",Strong,Strong,Strong,Strong,Unanimous
4,IQ-0005,1,Software Developer,"For a Software Developer interview, how would ...",I know unit testing is important in this role....,Strong,Average,Strong,Strong,Majority


## 1. Dataset and validation labels

`Final_Label` is based on the three validators. Unresolved rows are excluded from supervised training.

In [2]:
print('Records:', len(df))
print('Unique questions:', df.Question.nunique())
print('Unique answers:', df.Answer.nunique())
print('\nLabel distribution:')
print(df.Final_Label.value_counts())
print('\nAgreement:')
print(df.Validation_Agreement.value_counts())

Records: 1000
Unique questions: 20
Unique answers: 20

Label distribution:
Final_Label
Average       442
Strong        436
Weak          107
Unresolved     15
Name: count, dtype: int64

Agreement:
Validation_Agreement
Majority                                  809
Unanimous                                 171
Tie (Validator 3 Missing)                  15
2-of-2 Agreement (Validator 3 Missing)      5
Name: count, dtype: int64


## 2. Train and compare embeddings + fusion strategies

In [3]:
# Run the complete experiment from the project training script.
%run train_interviewiq.py

INTERVIEWIQ FINAL AI MODEL TRAINING

Records used: 985
Unique Questions: 20
Unique Answers: 20

Label distribution:
Final_Label
Average    442
Strong     436
Weak       107
Name: count, dtype: int64

Training records: 738
Testing records: 247

Building embedding representations...
  [1/4] TF-IDF + SVD complete
  [2/4] GloVe-style SVD complete
  [3/4] Word Distribution SVD complete
  [4/4] Subword SVD complete

Embedding: TFIDF-SVD

Training Early Fusion...
Early Fusion F1: 0.3303

Training Late Fusion...
Late Fusion F1: 0.3206

Embedding: GloVe-style-SVD

Training Early Fusion...
Early Fusion F1: 0.3303

Training Late Fusion...
Late Fusion F1: 0.3079

Embedding: Word-Distribution-SVD

Training Early Fusion...
Early Fusion F1: 0.3303

Training Late Fusion...
Late Fusion F1: 0.3079

Embedding: Subword-SVD

Training Early Fusion...
Early Fusion F1: 0.3274

Training Late Fusion...
Late Fusion F1: 0.3206


FINAL MODEL COMPARISON
            Embedding Fusion  Accuracy  Precision_macro  Recal

In [4]:
results = pd.read_csv('models/model_comparison.csv')
results.round(4)

,Embedding,Fusion,Accuracy,Precision_macro,Recall_macro,F1_macro
0,TFIDF-SVD,Early,0.3441,0.4019,0.4152,0.3303
1,GloVe-style-SVD,Early,0.3441,0.4019,0.4152,0.3303
2,Word-Distribution-SVD,Early,0.3441,0.4019,0.4152,0.3303
3,Subword-SVD,Early,0.3360,0.3925,0.4094,0.3274
4,Subword-SVD,Late,0.3401,0.3470,0.3569,0.3206
5,TFIDF-SVD,Late,0.3401,0.3470,0.3569,0.3206
6,GloVe-style-SVD,Late,0.3117,0.3667,0.3915,0.3079
7,Word-Distribution-SVD,Late,0.3117,0.3667,0.3915,0.3079


## 3. Best model

The experiment script evaluates multiple lightweight embedding representations using separate Question and Answer representations. Early Fusion and Late Fusion strategies are compared using an MLP neural classifier. The best measured configuration is saved to models/best_model.pkl and is used by the InterviewIQ web application for real-time answer evaluation.

In [5]:
import json
print(json.dumps(json.load(open('models/training_metadata.json')), indent=2))

{
  "records_used": 985,
  "unique_questions": 20,
  "unique_answers": 20,
  "train_records": 738,
  "test_records": 247,
  "best_embedding": "TFIDF-SVD",
  "best_fusion": "Early",
  "best_macro_f1": 0.3302898991005175,
  "architecture": "Separate Question/Answer Representations -> Early/Late Fusion -> MLP Neural Classifier"
}


## 4. Deployment test

In [6]:
from app.evaluator import evaluate_answer

question = "For a Software Developer interview, how would you explain object oriented programming?"

answer = "Object oriented programming uses classes and objects to organize code, support reuse, and model real-world entities."

keywords = ["class", "object", "inheritance", "encapsulation", "polymorphism"]

result = evaluate_answer(answer, question, [], keywords)

print(result)

InterviewIQ: trained model loaded successfully.
{'score': 100.0, 'label': 'Strong', 'feedback': 'Your answer covers the expected concept reasonably well. Try adding a concrete example when possible.'}
